<a href="https://colab.research.google.com/github/Feellived/molecular-reliability-signals/blob/yoonsoo/02_dedup_allowance_split.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 02 · 중복 제거 · 허용성 표 · 4중 분할 (팀 산출물 재사용)

**태스크 3, 7, 8** / **연구계획서 대응**: 5.2절, 5.7절

---

## 노트북 구성

| 노트북 | 하는 일 | 태스크 |
|---|---|---|
| 00 | `data/raw` 22종 확인, 변수 구조 점검, `data/processed` 현황 | 1 |
| 01 | 팀원 프로필을 표본 재계산해 대조 (선택) | 2·4·5·6 검증 |
| **02 (지금 여기)** | 중복 제거 → 허용성 표 → 4중 분할 | **3·7·8** |

## 이 노트북이 만들어진 이유

팀원이 이미 `data/processed/<물성명>/molecule_profile.csv` 로 **태스크 2·4·5·6을 끝내 두었다.**
같은 계산을 다시 할 이유가 없으므로, 그 결과를 읽어 쓰고 **남은 3·7·8만** 수행한다.

| 태스크 | 상태 | 출처 |
|---|---|---|
| 2 부모 분자 | ✅ 완료 | `molecule_profile.csv` → `smiles`, `valid`, `parent_smiles` |
| 4 골격 | ✅ 완료 | `molecule_profile.csv` → `scaffold` |
| 5 B1·B2·B3 | ✅ 완료(전수) | `n_tautomers`, `n_protomers`, `has_salt`, `has_stereo` |
| 6 비율·게이트 | ✅ 완료 | `prescreen_summary.csv`, `gate_decision_final.csv` |
| **3 중복 제거** | ❌ **세기만 함** | 이 노트북에서 수행 |
| **7 허용성 표** | ❌ | 이 노트북에서 수행 |
| **8 4중 분할** | ❌ | 이 노트북에서 수행 |

## 먼저 해결해야 하는 문제: 라벨이 없다

`molecule_profile.csv` 에는 구조 정보만 있고 **`Y`(정답값)와 `tdc_split`(원래 train_val인지 test인지)이 없다.**
분할을 하려면 둘 다 필요하므로 `data/raw/` 와 다시 이어붙인다.

> ### 조인 방법 — 왜 위치가 아니라 키로 붙이는가
> raw 안에 **같은 SMILES 문자열이 여러 번 등장**한다(ppbr_az 는 2790건 중 993건이 중복).
> 그런데 `molecule_profile.csv` 의 내용은 전부 **SMILES 문자열만의 함수**다 —
> 같은 문자열이면 부모 분자도 골격도 호변이성질체 수도 같다.
> 그래서 프로필을 `smiles` 기준으로 **유일화한 뒤 raw 각 행에 붙이면** 1:1로 안전하게 매칭된다.
> (행 순서가 같다고 가정할 필요가 없다.)

## 파일을 덮어쓰지 않는다

팀원 파일은 **읽기만** 한다. 우리 산출물은 새 이름으로 쓴다.

- `<물성명>/molecules_labeled.csv` — 프로필 + 라벨 + 중복 판정
- `<물성명>/splits.csv` — 4중 분할
- `reports/*.csv` — 집계표

In [ ]:
# ============================================================================
# [셀 1] 드라이브 마운트 + 경로  ← 모든 노트북 공통
# ----------------------------------------------------------------------------
# 읽는 곳과 쓰는 곳이 다르다.
#   data/processed 는 팀원이 공유한 폴더라 쓰기가 막혀 있다(Read-only file system).
#   팀 파일은 읽기만 하고, 우리 결과는 MIST/outputs 아래에 쓴다.
# 로컬에서 돌린다면 drive.mount 두 줄을 지우고 DATA_ROOT / OUT_ROOT 만 바꾸면 된다.
# ============================================================================
import warnings; warnings.filterwarnings("ignore")
from pathlib import Path
from google.colab import drive

drive.mount("/content/drive")

DATA_ROOT = Path("/content/drive/MyDrive/MIST/data")
RAW_DIR   = DATA_ROOT / "raw"             # [읽기] TDC 원본
TEAM_DIR  = DATA_ROOT / "processed"       # [읽기] 팀원 산출물

OUT_ROOT   = Path("/content/drive/MyDrive/MIST/outputs")
PROC_DIR   = OUT_ROOT / "processed"       # [쓰기] 물성별 우리 산출물
REPORT_DIR = PROC_DIR / "reports"         # [쓰기] 집계표
CACHE_DIR  = PROC_DIR / "_cache"          # [쓰기] 무거운 계산 캐시
for _d in (PROC_DIR, REPORT_DIR, CACHE_DIR):
    _d.mkdir(parents=True, exist_ok=True)

SEED = 42     # 분할·샘플링 재현용

print("[읽기] RAW  :", RAW_DIR, "(있음)" if RAW_DIR.exists() else "(없음 — 경로 확인)")
print("[읽기] TEAM :", TEAM_DIR, "(있음)" if TEAM_DIR.exists() else "(없음 — 경로 확인)")
print("[쓰기] OUT  :", PROC_DIR)

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
[읽기] RAW  : /content/drive/MyDrive/MIST/data/raw (있음)
[읽기] TEAM : /content/drive/MyDrive/MIST/data/processed (있음)
[쓰기] OUT  : /content/drive/MyDrive/MIST/outputs/processed


In [ ]:
# ============================================================================
# [셀 2] TDC ADMET Benchmark Group 22종 메타데이터 (참조용 상수)
# ----------------------------------------------------------------------------
# 연구계획서 5.1절: 회귀 9종 + 분류 13종 = 22종.
# 아래 dict 는 "우리가 알고 있는 정답"이며, 실제 데이터에서 자동 판별한 결과와
# 대조해서 틀리면 경고를 띄우는 용도로 쓴다(하드코딩만 믿지 않는다).
# ============================================================================
TASK_TYPE_REF = {
    # ---- 회귀 9종 (Y 가 연속값) ----
    "caco2_wang":                 "regression",   # Caco-2 세포 투과도 log(cm/s)
    "lipophilicity_astrazeneca":  "regression",   # logD (pH 7.4 옥탄올/물 분배계수)
    "solubility_aqsoldb":         "regression",   # logS 수용해도
    "ppbr_az":                    "regression",   # 혈장단백결합률 (%)
    "vdss_lombardo":              "regression",   # 정상상태 분포용적 (L/kg)
    "half_life_obach":            "regression",   # 반감기 (h)
    "clearance_hepatocyte_az":    "regression",   # 간세포 청소율 (uL/min/1e6 cells)
    "clearance_microsome_az":     "regression",   # 마이크로솜 청소율 (mL/min/g)
    "ld50_zhu":                   "regression",   # 급성독성 LD50 (-log mol/kg)
    # ---- 분류 13종 (Y 가 0/1) ----
    "hia_hou":                            "classification",  # 장 흡수 여부
    "pgp_broccatelli":                    "classification",  # P-gp 저해 여부
    "bioavailability_ma":                 "classification",  # 경구 생체이용률
    "bbb_martins":                        "classification",  # 혈뇌장벽 투과
    "cyp2c9_veith":                       "classification",  # CYP2C9 저해
    "cyp2d6_veith":                       "classification",  # CYP2D6 저해
    "cyp3a4_veith":                       "classification",  # CYP3A4 저해
    "cyp2c9_substrate_carbonmangels":     "classification",  # CYP2C9 기질
    "cyp2d6_substrate_carbonmangels":     "classification",  # CYP2D6 기질
    "cyp3a4_substrate_carbonmangels":     "classification",  # CYP3A4 기질
    "herg":                               "classification",  # hERG 심독성
    "ames":                               "classification",  # Ames 변이원성
    "dili":                               "classification",  # 약물유발 간손상
}
assert len(TASK_TYPE_REF) == 22
print("회귀:", sum(v == "regression" for v in TASK_TYPE_REF.values()),
      "/ 분류:", sum(v == "classification" for v in TASK_TYPE_REF.values()))

회귀: 9 / 분류: 13


## 1. 팀 프로필 + raw 라벨 조인

In [ ]:
# ============================================================================
# [셀 3] molecule_profile.csv 와 raw 를 이어붙여 '라벨 붙은 분자표' 만들기
# ============================================================================
import pandas as pd, numpy as np
from tqdm.auto import tqdm

pd.set_option("display.width", 250, "display.max_columns", 60)

DATASETS = sorted(TASK_TYPE_REF)           # 22종만. _tdc_cache, esol_pilot 등은 건드리지 않는다
PROFILE_COLS = ["smiles", "valid", "has_salt", "parent_smiles",
                "scaffold", "has_stereo", "n_tautomers", "n_protomers"]

tables, join_rows = {}, []

for name in tqdm(DATASETS, desc="조인"):
    # ---- raw: train_val + test 를 세로로 붙이고 출처를 기록 ----
    raw = pd.concat(
        [pd.read_csv(RAW_DIR / name / f).assign(tdc_split=s)
         for s, f in [("train_val", "train_val.csv"), ("test", "test.csv")]],
        ignore_index=True)

    # ---- 팀 프로필: smiles 기준 유일화 (같은 문자열이면 내용이 동일하므로 정보 손실 없음) ----
    prof = pd.read_csv(TEAM_DIR / name / "molecule_profile.csv")
    n_prof_raw = len(prof)
    prof_u = prof.drop_duplicates("smiles", keep="first")[PROFILE_COLS]

    # ---- left-merge: raw 각 행에 프로필을 붙인다 (행 수가 절대 늘지 않는다) ----
    d = raw.merge(prof_u, left_on="Drug", right_on="smiles", how="left", validate="many_to_one")
    assert len(d) == len(raw), "조인 후 행 수가 변했습니다"

    d = d.drop(columns=["smiles"]).rename(columns={"Drug": "smiles_original"})
    d.insert(0, "row_uid", [f"{name}__{i}" for i in range(len(d))])
    d["dataset"] = name
    d["task_type"] = TASK_TYPE_REF[name]

    tables[name] = d
    join_rows.append({
        "dataset": name, "n_raw": len(raw),
        "n_profile": n_prof_raw, "n_profile_unique": len(prof_u),
        "매칭실패": int(d.parent_smiles.isna().sum()),      # 0 이어야 정상
        "valid_False": int((d.valid == False).sum()),
        "골격_빈값": int(d.scaffold.isna().sum()),          # 고리 없는 분자 (정상)
    })

join_report = pd.DataFrame(join_rows)
display(join_report)

bad = join_report[join_report.매칭실패 > 0]
assert len(bad) == 0, f"프로필을 못 찾은 행이 있습니다:\n{bad}"
print(f"✅ 조인 성공 — 총 {join_report.n_raw.sum():,}행")
print(f"   (고리 없는 분자 = 골격 빈값: {join_report.골격_빈값.sum():,}행. 이건 오류가 아니다)")

조인:   0%|          | 0/22 [00:00<?, ?it/s]

,dataset,n_raw,n_profile,n_profile_unique,매칭실패,valid_False,골격_빈값
0,ames,7278,7278,7255,0,0,1268
1,bbb_martins,2030,2030,1975,0,0,99
2,bioavailability_ma,640,640,640,0,0,17
3,caco2_wang,910,910,906,0,0,25
4,clearance_hepatocyte_az,1213,1213,1020,0,0,4
5,clearance_microsome_az,1102,1102,1102,0,0,1
6,cyp2c9_substrate_carbonmangels,669,669,666,0,0,21
7,cyp2c9_veith,12092,12092,12092,0,0,272
8,cyp2d6_substrate_carbonmangels,667,667,664,0,0,21
9,cyp2d6_veith,13130,13130,13130,0,0,275


✅ 조인 성공 — 총 81,809행
   (고리 없는 분자 = 골격 빈값: 7,346행. 이건 오류가 아니다)


In [ ]:
# ============================================================================
# [셀 4] 01 노트북의 검증 결과 확인 (있으면 요약, 없으면 안내)
# ----------------------------------------------------------------------------
# 팀 프로필을 그대로 믿고 쓰는 노트북이므로, 검증이 돌아갔는지만 확인하고 넘어간다.
# 실제 재계산은 01_verify_team_profile.ipynb 이 담당한다(무거운 패키지가 거기 있다).
# ============================================================================
VER = REPORT_DIR / "09_verification_columns.csv"

if VER.exists():
    v = pd.read_csv(VER)
    print("01_verify_team_profile 검증 결과 (가중평균 일치율 %)")
    for c in ["부모분자", "골격", "염", "입체", "호변_판정", "양성자_판정"]:
        if c in v.columns:
            print(f"  {c:12s} {(v[c] * v.n).sum() / v.n.sum():6.1f}")
    struct = [c for c in ["부모분자", "골격", "염", "입체"] if c in v.columns]
    low = v[(v[struct] < 99).any(axis=1)]
    print()
    print("구조 열이 99% 미만인 물성:", low.dataset.tolist() or "없음")
else:
    print("[알림] 검증 리포트가 없습니다 — 01_verify_team_profile.ipynb 을 아직 안 돌렸습니다.")
    print("       팀 프로필을 그대로 신뢰하고 진행합니다. 분할 결과 자체는 영향받지 않습니다.")
    print("       다만 게이트 판정(어떤 변형 축을 쓸지)이 논문 근거이므로,")
    print("       최종 결과를 내기 전에 01 을 한 번은 돌려 두세요.")

01_verify_team_profile 검증 결과 (가중평균 일치율 %)
  부모분자           99.5
  골격             99.7
  염             100.0
  입체            100.0
  호변_판정          99.9
  양성자_판정        100.0

구조 열이 99% 미만인 물성: ['bioavailability_ma', 'cyp2c9_veith', 'hia_hou', 'solubility_aqsoldb']


### 📌 사전 확정 사항 — 팀원 `parent_smiles` · `scaffold` 를 그대로 쓴다 (2026-08-17 확정)

노트북 01의 독립 재계산 결과 일치율은 부모분자 **99.5%**, 골격 **99.7%**,
염·입체 **100%**, 호변/양성자 판정 **99.9 / 100%** 였다. 팀 산출물은 신뢰할 수 있다.

불일치 0.5% 는 무작위 오류가 아니라 **한 가지 원인**이었다 — **술폭사이드 표기**.

```
팀원: ...S(=O)...        중성 표기 (S=O 이중결합)
우리: ...[S+]([O-])...   전하 분리 표기 (S+-O- 단일결합)
```

RDKit `Cleanup()` 의 정규화 규칙이 술폭사이드를 전하 분리형으로 바꾼다.
오메프라졸 계열, 설린닥, 모다피닐처럼 술폭사이드를 가진 약물이 여기 걸린다.

**팀원 표기를 채택하는 이유는 골격 때문이다.** Murcko 는 이중결합으로 붙은 곁원자만 남기므로,
전하 분리형에서는 S+-O- 가 단일결합이 되어 **산소가 잘려나가고 `[SH+]` 만 남는다.**

```
팀원 골격: O=S(Cc1ccccn1)c1nc2ccccc2[nH]1      <- 계열을 대표함
우리 골격: c1ccc(C[SH+]c2nc3ccccc3[nH]2)nc1    <- 화학적으로 기괴함
```

> ### ⚠️ 다음 단계 규칙
> 변형을 생성할 때 `parent_smiles` 에 **`rdMolStandardize.Cleanup()` 을 다시 걸지 않는다.**
> 거는 순간 술폭사이드가 전하 분리형으로 바뀌어 지문·골격·예측이 전부 달라진다.
> 필요한 건 `Chem.MolFromSmiles(parent_smiles)` 까지다.

### 열거 설정 차이 (게이트에는 무해, 변형 생성에는 중요)

개수 자체는 양쪽 다 **한 방향으로만** 갈렸다 = 구현 차이가 아니라 설정 차이다.

| 축 | 같음 | 우리가 많음 | 우리가 적음 | 해석 |
|---|---|---|---|---|
| 호변이성질체 | 94.0% | 5.9% | 0.1% | 팀원이 더 낮은 `maxTautomers` 또는 구버전 변환 규칙 |
| 양성자화 | 82.7% | 0.0% | 17.2% | 팀원이 더 넓은 pH 창 또는 높은 `pka_precision` |

"2개 이상인가" 판정은 99.9~100% 일치하므로 **게이트(계획서 5.4절)에는 영향이 없다.**
다만 변형 생성 단계에서는 **개수가 곧 B1 값**이 되므로, 그때 pH 창과 열거 상한을
명시적으로 고정하고 문서에 적어야 한다 (현재 노트북 01 기준: pH 6.4~8.4,
`pka_precision=1.0`, `maxTautomers=32`).

## 2. 태스크 3 — 부모 분자 기준 중복 제거

팀원 표는 중복을 **세기만** 했다. 실제로 지우는 건 여기서 한다.

**중복 판정 키**: `parent_smiles` (탈염된 부모 분자)

**남길 행** (`is_kept`)
1. `tdc_split == "test"` 인 행 우선 — TDC 공식 시험 집합 소속 정보를 잃지 않기 위해서
2. 그다음은 등장 순서가 빠른 행

**Y가 서로 다를 때(라벨 충돌)**

| 과제 | 정책 | 이유 |
|---|---|---|
| 회귀 | 평균으로 합침 | 같은 화합물의 반복 측정으로 보는 것이 자연스럽다 |
| 분류 | 충돌 그룹 전체 제거 | 0과 1은 평균낼 수 없고 어느 쪽이 맞는지 알 수 없다 |

`ppbr_az`(35.6%)와 `clearance_hepatocyte_az`(15.9%)는 손실이 크므로 결과를 꼭 확인할 것.

In [ ]:
# ============================================================================
# [셀 5] 중복 제거
# ============================================================================
DUP_POLICY_REG = "mean"   # 회귀 라벨 충돌: "mean" 또는 "drop"
DUP_POLICY_CLF = "drop"   # 분류 라벨 충돌: "drop" 권장

def dedup_by_parent(df, task_type):
    d = df.copy()
    d["Y"] = pd.to_numeric(d["Y"], errors="coerce")
    d["dup_role"] = "unique"
    d["is_kept"] = True
    d["Y_final"] = d["Y"]
    d["dup_group_size"] = 1

    # 파싱 실패(valid=False)나 프로필 없음은 중복 판정 대상이 아니다
    invalid = (d.valid != True) | d.parent_smiles.isna()
    d.loc[invalid, "is_kept"] = False
    d.loc[invalid, "dup_role"] = "invalid"

    valid = d[~invalid]
    # 남길 행 우선순위: test 소속이면 0, 아니면 1 (작을수록 우선)
    prio = np.where(valid["tdc_split"].values == "test", 0, 1)
    valid = valid.assign(_prio=prio).sort_values("_prio", kind="stable")

    n_removed = n_conflict_groups = n_dropped = 0

    for _, idx in valid.groupby("parent_smiles", sort=False).groups.items():
        idx = list(idx)
        if len(idx) == 1:
            continue
        ys = d.loc[idx, "Y"]
        keeper, others = idx[0], idx[1:]
        d.loc[idx, "dup_group_size"] = len(idx)

        same_label = bool(np.allclose(ys.values, ys.values[0], rtol=0, atol=1e-9))
        if same_label:
            d.loc[keeper, "dup_role"] = "kept"
            d.loc[others, "is_kept"] = False
            d.loc[others, "dup_role"] = "removed_duplicate"
            n_removed += len(others)
        else:
            n_conflict_groups += 1
            if task_type == "regression" and DUP_POLICY_REG == "mean":
                d.loc[keeper, "Y_final"] = float(ys.mean())
                d.loc[keeper, "dup_role"] = "kept_label_averaged"
                d.loc[others, "is_kept"] = False
                d.loc[others, "dup_role"] = "removed_duplicate"
                n_removed += len(others)
            else:
                d.loc[idx, "is_kept"] = False
                d.loc[idx, "dup_role"] = "dropped_label_conflict"
                n_dropped += len(idx)

    return d, {"n_removed_duplicate": n_removed,
               "n_conflict_groups": n_conflict_groups,
               "n_dropped_conflict": n_dropped}


dedup_rows = []
for name in tqdm(DATASETS, desc="중복 제거"):
    tables[name], stat = dedup_by_parent(tables[name], TASK_TYPE_REF[name])
    d = tables[name]
    dedup_rows.append({
        "dataset": name, "task_type": TASK_TYPE_REF[name],
        "n_before": len(d),
        "n_invalid": int((d.dup_role == "invalid").sum()),
        **stat,
        "n_after": int(d.is_kept.sum()),
        "pct_lost": round(100 * (1 - d.is_kept.mean()), 2),
    })

dedup_report = pd.DataFrame(dedup_rows).sort_values("pct_lost", ascending=False)
display(dedup_report)
print(f"전체: {dedup_report.n_before.sum():,} -> {dedup_report.n_after.sum():,} 건 "
      f"({dedup_report.n_before.sum() - dedup_report.n_after.sum():,} 감소)")

중복 제거:   0%|          | 0/22 [00:00<?, ?it/s]

,dataset,task_type,n_before,n_invalid,n_removed_duplicate,n_conflict_groups,n_dropped_conflict,n_after,pct_lost
19,ppbr_az,regression,2790,0,993,691,0,1797,35.59
4,clearance_hepatocyte_az,regression,1213,0,193,193,0,1020,15.91
20,solubility_aqsoldb,regression,9982,0,406,184,0,9576,4.07
1,bbb_martins,classification,2030,0,53,11,22,1955,3.69
14,herg,classification,655,0,10,3,6,639,2.44
21,vdss_lombardo,regression,1130,0,19,17,0,1111,1.68
8,cyp2d6_substrate_carbonmangels,classification,667,0,2,1,2,663,0.60
16,ld50_zhu,regression,7385,0,43,30,0,7342,0.58
3,caco2_wang,regression,910,0,5,5,0,905,0.55
18,pgp_broccatelli,classification,1218,0,6,0,0,1212,0.49


전체: 81,809 -> 79,903 건 (1,906 감소)


In [ ]:
# ============================================================================
# [셀 6] 팀원이 센 중복 수와 우리가 실제로 지운 수 비교 + 라벨 충돌 사례
# ============================================================================
pre = pd.read_csv(TEAM_DIR / "prescreen_summary.csv")[
    ["dataset", "n_total", "n_dup_raw_smiles", "n_dup_parent_molecules"]]
cmp = pre.merge(dedup_report[["dataset", "n_removed_duplicate", "n_dropped_conflict",
                              "n_conflict_groups", "n_after"]], on="dataset")
cmp["센것_지운것_차이"] = cmp.n_dup_parent_molecules - (cmp.n_removed_duplicate + cmp.n_dropped_conflict)
display(cmp)
print("차이가 0이 아닌 경우: 라벨 충돌 그룹을 통째로 버려서(분류) 더 많이 지웠거나,")
print("반대로 팀원 집계 기준(중복 '개수' 정의)이 달라서 생긴다. 위 표로 그 차이를 명시해 둔다.")

print("\n[라벨 충돌 사례] 같은 부모 분자인데 Y 가 다른 경우")
shown = 0
for name, d in tables.items():
    conf = d[d.dup_role.isin(["dropped_label_conflict", "kept_label_averaged"])]
    if not len(conf):
        continue
    key = conf.parent_smiles.iloc[0]
    g = d[d.parent_smiles == key]
    print(f"\n  [{name}] ({TASK_TYPE_REF[name]}) 부모: {key[:70]}")
    for _, r in g.iterrows():
        print(f"    Y={r.Y!s:>10}  ->  Y_final={r.Y_final!s:>10}  {r.dup_role}")
    shown += 1
    if shown >= 4:
        break

,dataset,n_total,n_dup_raw_smiles,n_dup_parent_molecules,n_removed_duplicate,n_dropped_conflict,n_conflict_groups,n_after,센것_지운것_차이
0,ames,7278,23,23,23,0,0,7255,0
1,bbb_martins,2030,55,64,53,22,11,1955,-11
2,bioavailability_ma,640,0,0,0,0,0,640,0
3,caco2_wang,910,4,5,5,0,5,905,0
4,clearance_hepatocyte_az,1213,193,193,193,0,193,1020,0
5,clearance_microsome_az,1102,0,0,0,0,0,1102,0
6,cyp2c9_substrate_carbonmangels,669,3,3,3,0,0,666,0
7,cyp2c9_veith,12092,0,39,30,15,6,12047,-6
8,cyp2d6_substrate_carbonmangels,667,3,3,2,2,1,663,-1
9,cyp2d6_veith,13130,0,37,33,8,4,13089,-4


차이가 0이 아닌 경우: 라벨 충돌 그룹을 통째로 버려서(분류) 더 많이 지웠거나,
반대로 팀원 집계 기준(중복 '개수' 정의)이 달라서 생긴다. 위 표로 그 차이를 명시해 둔다.

[라벨 충돌 사례] 같은 부모 분자인데 Y 가 다른 경우

  [bbb_martins] (classification) 부모: CC(=O)Oc1ccccc1C(=O)O
    Y=         0  ->  Y_final=         0  dropped_label_conflict
    Y=         1  ->  Y_final=         1  dropped_label_conflict

  [caco2_wang] (regression) 부모: NC(N)=N/N=C/c1c(Cl)cccc1Cl
    Y=-4.6799998  ->  Y_final=-4.52170585  kept_label_averaged
    Y=-4.363411900000001  ->  Y_final=-4.363411900000001  removed_duplicate

  [clearance_hepatocyte_az] (regression) 부모: O=C(O)CCc1ccc(OCc2cc(Cl)ccc2-c2ccccc2)cc1
    Y=      13.8  ->  Y_final=     17.59  kept_label_averaged
    Y=     21.38  ->  Y_final=     21.38  removed_duplicate

  [cyp2c9_veith] (classification) 부모: CN1CCC[C@@H]1c1cccnc1
    Y=         1  ->  Y_final=         1  dropped_label_conflict
    Y=         0  ->  Y_final=         0  dropped_label_conflict
    Y=         0  ->  Y_final=         0  dropped_label_conflict
    Y=     

## 3. 태스크 7 — 물성별 변형 허용성 표

모든 물성에 모든 변형이 의미 있는 게 아니다.

- **logD (pH 7.4)** 처럼 정의 자체에 pH가 들어간 물성 → 양성자화 변형이 **본질적으로 의미 있다**.
- **수용해도**처럼 중성종 기준으로 보고되는 물성 → 이온형 강제는 **라벨 정의와 충돌**한다.
- **LD50** 은 투여 질량 기준이라 탈염(B2)이 분자량을 통해 라벨의 의미를 바꾼다.

논문으로 참고할 게 없고 TDC의 물성 설명을 직접 읽어야 하는 **수작업**이다.
아래는 초안이며, CSV로 떨어뜨려 **직접 고친 뒤 다시 불러오는** 구조다
(파일이 이미 있으면 덮어쓰지 않는다).

| 값 | 의미 |
|---|---|
| `허용` | 화학적으로 말이 된다. 주 분석에 사용 |
| `주의` | 조건부. 쓰되 해석에 단서를 단다 |
| `충돌` | 라벨 정의와 충돌한다. 이 물성에서는 그 축을 쓰지 않는다 |

### 📌 사전 확정 사항 — 양성자화 상태의 정의 (2026-08-17 확정)

**B1-양성자화의 상태 집합에 "등록된 형태"를 포함한다.**

`dimorphite-dl` 이 돌려주는 건 지정 pH 창에서 존재할 법한 상태들뿐이고, 등록형은 거기 없을 수 있다.
예: 아세트산 `CC(=O)O` 는 pH 6.4~8.4 에서 `CC(=O)[O-]` **하나만** 반환한다.
열거 결과만 세면 "상태 1개 = 변형 불가"가 되지만, 실제로는
**등록형(중성) ↔ 우세형(음이온)** 이라는 비교 대상이 존재한다.

계획서 4.1절이 문제 삼는 것이 바로 "어느 상태로 등록되어 있느냐가 데이터 출처에 따라 달라지며
사용자가 그 선택을 통제하지 못한다"이고, 배포 시 모델이 실제로 받는 입력도 등록된 문자열이다.
따라서 등록형을 상태 집합에 넣는 쪽이 연구 질문에 충실하다.

**실측 근거** (노트북 01, pH 6.4~8.4, 물성당 200개 표본)

| 정의 | 중앙값 | 10% 게이트 통과 |
|---|---|---|
| 열거 결과만 (팀원 정의) | 88.5% | 22/22 |
| 등록형 포함 (채택) | 91.1% | 22/22 |

- **게이트 판정은 두 정의가 동일**하므로 계획서 5.4절의 축 선택은 영향받지 않는다.
- 차이가 큰 물성: `clearance_microsome_az` +7.4%p, `ppbr_az` +2.7%p, `lipophilicity_astrazeneca` +2.8%p
  (중성형으로 등록된 카복실산·아민이 많은 데이터)
- 차이가 작은 물성: `vdss_lombardo` +0.8%p, `herg` +1.0%p
  (이미 이온형으로 등록된 비율이 높다 — 각각 부모 분자의 73%, 54%가 전하 보유)
- `ames` · `ld50_zhu` · `solubility_aqsoldb` 는 분자의 절반이 **진짜 비이온성**이라
  어느 정의를 써도 50%대에 머문다. 계획서 5.1절 심층 분석 대상 선정에서 이 점을 고려할 것.

**적용 지점**: 이 노트북은 게이트만 다루므로 영향이 없다.
다음 단계인 **변형 생성**에서 상태 집합을 `protomer_states(parent) | {parent}` 로 구성한다.

**단서**: 위 수치는 pH 6.4~8.4 창 기준이다. 아래 허용성 표에서 `hia_hou` 의 pH 창을
위장관 범위로 넓히기로 결정하면 그 물성은 다시 재야 한다.

In [ ]:
# ============================================================================
# [셀 7] 변형 허용성 표 초안 (수작업 산출물)
# ----------------------------------------------------------------------------
# 열 의미
#   assay_ph        : 측정이 이루어지는 pH 조건 (양성자화 변형의 타당성을 좌우한다)
#   B1_tautomer     : 호변이성질체 변형 허용 여부
#   B1_protonation  : 양성자화 상태 변형 허용 여부
#   B2_salt         : 원본 염 형태 <-> 탈염 형태 비교 허용 여부
#   B3_stereo       : 입체 표기 유지 <-> 제거 비교 허용 여부
# ============================================================================
import pandas as pd, numpy as np

A, C, X = "허용", "주의", "충돌"

ALLOWANCE_DRAFT = [
    # dataset, assay_ph, B1_taut, B1_prot, B2_salt, B3_stereo, 근거
    ("caco2_wang", "정단 6.5 / 기저 7.4", A, A, A, A,
     "Caco-2 단층 투과도. pH 구배 조건에서 측정되므로 양성자화 상태가 투과 기전에 직접 관여한다."),
    ("lipophilicity_astrazeneca", "7.4", A, A, A, A,
     "logD7.4 는 정의에 pH 7.4 가 포함된다. 양성자화 변형이 가장 본질적인 물성."),
    ("solubility_aqsoldb", "무완충/출처마다 상이", A, C, X, A,
     "고유 수용해도는 대개 중성종 기준으로 보고되고 완충 조건이 통일돼 있지 않다. 이온형 강제는 라벨 정의와 충돌 소지. "
     "염 형태 자체가 용해도를 크게 바꾸므로 탈염은 측정 대상 물질을 바꾸는 셈 -> B2 충돌."),
    ("ppbr_az", "7.4 (혈장)", A, A, A, A,
     "혈장 단백 결합률. 혈장 pH 7.4 에서 정의되며 이온형이 알부민 결합에 직접 영향."),
    ("vdss_lombardo", "7.4 (생체)", A, A, A, A,
     "정상상태 분포용적. 생리적 pH 에서의 이온화가 조직 분배를 좌우한다."),
    ("half_life_obach", "7.4 (생체)", A, A, A, A,
     "생체 내 반감기. 생리적 pH 기준."),
    ("clearance_hepatocyte_az", "7.4 (완충액)", A, A, A, A,
     "간세포 청소율. 배양 완충액 pH 7.4."),
    ("clearance_microsome_az", "7.4 (완충액)", A, A, A, A,
     "마이크로솜 청소율. 인큐베이션 완충액 pH 7.4."),
    ("ld50_zhu", "생체 (경구/복강)", A, C, X, A,
     "급성독성 LD50 은 투여 질량을 몰 기준으로 환산한 값이라 염 형태가 분자량을 통해 라벨에 직접 들어간다 -> B2 충돌."),
    ("hia_hou", "위장관 1.5~8", A, C, A, A,
     "장 흡수. 위장관 pH 폭이 커서 6.4~8.4 단일 창으로는 부족하다. pH 범위를 넓혀 재열거할지 검토 필요."),
    ("pgp_broccatelli", "7.4", A, A, A, A,
     "P-gp 저해 세포 기반 assay. pH 7.4."),
    ("bioavailability_ma", "생체 (경구)", A, C, C, A,
     "경구 생체이용률은 제형/염 형태의 영향을 크게 받는다. B2 는 사용하되 해석에 단서 필요."),
    ("bbb_martins", "7.4", A, A, A, A,
     "혈뇌장벽 투과. 생리적 pH 에서 중성/이온형 비율이 수동확산을 좌우한다."),
    ("cyp2c9_veith", "7.4 (완충액)", A, A, A, A, "CYP2C9 저해 형광 assay. 완충액 pH 7.4."),
    ("cyp2d6_veith", "7.4 (완충액)", A, A, A, A, "CYP2D6 저해. 염기성 아민의 양이온형이 결합에 중요."),
    ("cyp3a4_veith", "7.4 (완충액)", A, A, A, A, "CYP3A4 저해. 완충액 pH 7.4."),
    ("cyp2c9_substrate_carbonmangels", "7.4", A, A, A, A, "CYP2C9 기질 여부. 대사 assay pH 7.4."),
    ("cyp2d6_substrate_carbonmangels", "7.4", A, A, A, A, "CYP2D6 기질 여부. 대사 assay pH 7.4."),
    ("cyp3a4_substrate_carbonmangels", "7.4", A, A, A, A, "CYP3A4 기질 여부. 대사 assay pH 7.4."),
    ("herg", "7.4 (전기생리)", A, A, A, A,
     "hERG 차단. 염기성 아민의 양이온형이 채널 내강 결합의 핵심 기전이라 양성자화 변형이 특히 의미 있다."),
    ("ames", "~7.4 (S9 mix)", A, A, A, A,
     "세균 변이원성. 니트로/아미노 방향족의 호변이성질체 표기가 출처마다 갈리는 대표적 사례."),
    ("dili", "임상 (약물 단위)", A, C, C, A,
     "약물유발 간손상. 의약품 라벨 기준이라 염 형태가 곧 제품 형태다. B2 해석에 단서 필요."),
]

allow_draft = pd.DataFrame(
    ALLOWANCE_DRAFT,
    columns=["dataset", "assay_ph", "B1_tautomer", "B1_protonation",
             "B2_salt", "B3_stereo", "rationale"])
allow_draft.insert(1, "task_type", allow_draft.dataset.map(TASK_TYPE_REF))
# TDC 공식 설명 문서 링크 (직접 읽고 검토할 때 쓴다)
TOX_SETS = {"ld50_zhu", "herg", "ames", "dili"}
allow_draft["tdc_doc"] = np.where(
    allow_draft.dataset.isin(TOX_SETS),
    "https://tdcommons.ai/single_pred_tasks/tox",
    "https://tdcommons.ai/single_pred_tasks/adme")
allow_draft["reviewed"] = False     # <- 직접 확인한 물성부터 True 로 바꿔 나간다

assert len(allow_draft) == 22 and allow_draft.dataset.is_unique
display(allow_draft[["dataset", "task_type", "assay_ph",
                     "B1_tautomer", "B1_protonation", "B2_salt", "B3_stereo"]])

,dataset,task_type,assay_ph,B1_tautomer,B1_protonation,B2_salt,B3_stereo
0,caco2_wang,regression,정단 6.5 / 기저 7.4,허용,허용,허용,허용
1,lipophilicity_astrazeneca,regression,7.4,허용,허용,허용,허용
2,solubility_aqsoldb,regression,무완충/출처마다 상이,허용,주의,충돌,허용
3,ppbr_az,regression,7.4 (혈장),허용,허용,허용,허용
4,vdss_lombardo,regression,7.4 (생체),허용,허용,허용,허용
5,half_life_obach,regression,7.4 (생체),허용,허용,허용,허용
6,clearance_hepatocyte_az,regression,7.4 (완충액),허용,허용,허용,허용
7,clearance_microsome_az,regression,7.4 (완충액),허용,허용,허용,허용
8,ld50_zhu,regression,생체 (경구/복강),허용,주의,충돌,허용
9,hia_hou,classification,위장관 1.5~8,허용,주의,허용,허용


In [ ]:
# ============================================================================
# [셀 8] 허용성 표 저장/불러오기 — 손으로 고친 내용을 덮어쓰지 않는다
# ----------------------------------------------------------------------------
# 이미 파일이 있으면 그 파일을 신뢰하고 불러온다(= 당신이 검토해 고쳐 둔 버전).
# 초안으로 되돌리려면 OVERWRITE_ALLOWANCE = True 로 두고 한 번 실행.
# ============================================================================
OVERWRITE_ALLOWANCE = False
ALLOW_PATH = REPORT_DIR / "06_transformation_allowance.csv"

if ALLOW_PATH.exists() and not OVERWRITE_ALLOWANCE:
    allowance = pd.read_csv(ALLOW_PATH)
    print(f"기존 허용성 표를 불러왔습니다 (수동 편집 보존): {ALLOW_PATH}")
else:
    allowance = allow_draft.copy()
    allowance.to_csv(ALLOW_PATH, index=False)
    print(f"초안을 저장했습니다: {ALLOW_PATH}")

n_reviewed = int(allowance.reviewed.astype(bool).sum())
print(f"\n검토 완료 표시된 물성: {n_reviewed} / 22")
if n_reviewed < 22:
    print("  [할 일] 위 CSV 를 열어 tdc_doc 링크의 물성 설명을 읽고,")
    print("          각 축의 허용/주의/충돌 값을 확정한 뒤 reviewed 를 True 로 바꾸세요.")
    print("          계획서 5.2절: 이 표는 결과를 보기 전에 고정해야 합니다.")

print("\n축별 판정 분포")
for c in ["B1_tautomer", "B1_protonation", "B2_salt", "B3_stereo"]:
    print(f"  {c:16s}", allowance[c].value_counts().to_dict())

기존 허용성 표를 불러왔습니다 (수동 편집 보존): /content/drive/MyDrive/MIST/outputs/processed/reports/06_transformation_allowance.csv

검토 완료 표시된 물성: 0 / 22
  [할 일] 위 CSV 를 열어 tdc_doc 링크의 물성 설명을 읽고,
          각 축의 허용/주의/충돌 값을 확정한 뒤 reviewed 를 True 로 바꾸세요.
          계획서 5.2절: 이 표는 결과를 보기 전에 고정해야 합니다.

축별 판정 분포
  B1_tautomer      {'허용': 22}
  B1_protonation   {'허용': 17, '주의': 5}
  B2_salt          {'허용': 18, '충돌': 2, '주의': 2}
  B3_stereo        {'허용': 22}


## 4. 게이트(팀원 수치) × 허용성 = 최종 사용 축

팀원의 `gate_decision_final.csv` 는 B1을 "미소상태" 하나로 합쳐 두었다.
계획서 5.4절은 호변이성질체와 양성자화를 **따로** 세라고 하므로,
`prescreen_summary.csv` 의 `pct_multi_tautomer` / `pct_multi_protomer` 로 분리해 다시 만든다.
그리고 팀원의 생존 판정과 어긋나는 곳이 있는지 대조한다.

In [ ]:
# ============================================================================
# [셀 9] 게이트 표 재구성 (팀 수치 재사용, B1 을 두 축으로 분리) + 허용성과 교차
# ============================================================================
GATE_THRESHOLD = 10.0     # 계획서 5.4절: 10% 미만이면 주 분석에서 제외

pre = pd.read_csv(TEAM_DIR / "prescreen_summary.csv")
gate = pre[["dataset", "n_total", "pct_multi_tautomer", "pct_multi_protomer",
            "pct_with_salt", "pct_with_stereo"]].rename(columns={
    "pct_multi_tautomer": "pct_B1_tautomer",
    "pct_multi_protomer": "pct_B1_protomer",
    "pct_with_salt":      "pct_B2_salt",
    "pct_with_stereo":    "pct_B3_stereo"})

for axis in ["B1_tautomer", "B1_protomer", "B2_salt", "B3_stereo"]:
    gate[f"gate_{axis}"] = np.where(gate[f"pct_{axis}"] >= GATE_THRESHOLD, "주분석", "보조관찰")

# ---- 팀원 판정과 대조 ----
team = pd.read_csv(TEAM_DIR / "gate_decision_final.csv")
chk = gate.merge(team[["dataset", "B1_생존", "B2_생존", "B3_생존"]], on="dataset")
chk["B1_우리"] = (chk.gate_B1_tautomer == "주분석") | (chk.gate_B1_protomer == "주분석")
mism = chk[(chk.B1_우리 != chk.B1_생존) |
           ((chk.gate_B2_salt == "주분석") != chk.B2_생존) |
           ((chk.gate_B3_stereo == "주분석") != chk.B3_생존)]
print("팀원 생존 판정과 어긋나는 물성:", mism.dataset.tolist() or "없음")

print("\n축별 통과 물성 수 (10% 기준)")
for axis in ["B1_tautomer", "B1_protomer", "B2_salt", "B3_stereo"]:
    v = gate[f"pct_{axis}"]
    print(f"  {axis:12s} 통과 {int((v >= GATE_THRESHOLD).sum()):2d}/22   "
          f"중앙값 {v.median():5.1f}%  범위 {v.min():.1f}~{v.max():.1f}%")

# ---- 허용성과 교차: 두 조건을 모두 만족해야 실제로 쓴다 ----
AXIS_PAIRS = [("B1_tautomer", "gate_B1_tautomer", "B1_tautomer"),
              ("B1_protonation", "gate_B1_protomer", "B1_protonation"),
              ("B2_salt", "gate_B2_salt", "B2_salt"),
              ("B3_stereo", "gate_B3_stereo", "B3_stereo")]

dec = allowance[["dataset", "task_type"]].merge(
    gate[["dataset"] + [g for _, g, _ in AXIS_PAIRS]], on="dataset", how="left")
allowed = allowance.set_index("dataset")

for axis, gcol, acol in AXIS_PAIRS:
    dec[f"use_{axis}"] = [
        "사용" if (dec.loc[i, gcol] == "주분석" and allowed.loc[dec.loc[i, "dataset"], acol] in ("허용", "주의"))
        else ("허용안됨" if allowed.loc[dec.loc[i, "dataset"], acol] == "충돌" else "표본부족")
        for i in dec.index]

use_cols = [f"use_{a}" for a, _, _ in AXIS_PAIRS]
display(dec[["dataset", "task_type"] + use_cols])
for c in use_cols:
    print(f"  {c:20s} 사용 {int((dec[c]=='사용').sum()):2d}/22  "
          f"(허용안됨 {int((dec[c]=='허용안됨').sum())}, 표본부족 {int((dec[c]=='표본부족').sum())})")

gate.to_csv(REPORT_DIR / "05_variation_gate.csv", index=False)
dec.to_csv(REPORT_DIR / "07_axis_decision.csv", index=False)
print(f"\n저장: 05_variation_gate.csv (출처=팀 prescreen_summary), 07_axis_decision.csv")

팀원 생존 판정과 어긋나는 물성: 없음

축별 통과 물성 수 (10% 기준)
  B1_tautomer  통과 22/22   중앙값  68.6%  범위 44.4~86.4%
  B1_protomer  통과 22/22   중앙값  88.5%  범위 46.5~95.0%
  B2_salt      통과  1/22   중앙값   0.0%  범위 0.0~11.0%
  B3_stereo    통과 19/22   중앙값  36.7%  범위 0.0~59.9%


,dataset,task_type,use_B1_tautomer,use_B1_protonation,use_B2_salt,use_B3_stereo
0,caco2_wang,regression,사용,사용,표본부족,사용
1,lipophilicity_astrazeneca,regression,사용,사용,표본부족,사용
2,solubility_aqsoldb,regression,사용,사용,허용안됨,표본부족
3,ppbr_az,regression,사용,사용,표본부족,사용
4,vdss_lombardo,regression,사용,사용,표본부족,사용
5,half_life_obach,regression,사용,사용,표본부족,사용
6,clearance_hepatocyte_az,regression,사용,사용,표본부족,사용
7,clearance_microsome_az,regression,사용,사용,표본부족,사용
8,ld50_zhu,regression,사용,사용,허용안됨,표본부족
9,hia_hou,classification,사용,사용,표본부족,사용


  use_B1_tautomer      사용 22/22  (허용안됨 0, 표본부족 0)
  use_B1_protonation   사용 22/22  (허용안됨 0, 표본부족 0)
  use_B2_salt          사용  0/22  (허용안됨 2, 표본부족 20)
  use_B3_stereo        사용 19/22  (허용안됨 0, 표본부족 3)

저장: 05_variation_gate.csv (출처=팀 prescreen_summary), 07_axis_decision.csv


### 📌 축 제외 근거 기록 (2026-08-17)

게이트 x 허용성 교차 결과 실제로 쓰는 축은 다음과 같다.

| 축 | 사용 | 비고 |
|---|---|---|
| B1 호변이성질체 | **22 / 22** | 전 물성 생존 |
| B1 양성자화 | **22 / 22** | 전 물성 생존 |
| B2 염 형태 | **0 / 22** | 허용안됨 2 (라벨 정의 충돌) + 표본부족 20 |
| B3 입체 표기 | **19 / 22** | 표본부족 3 |

#### B2 가 전멸한 것은 계획서가 예상한 시나리오다

> 5.4절: B2와 B3는 표준화의 결과로 소멸할 가능성이 실재하며, **두 축이 모두 기준에 미달할 경우**
> B는 B1을 중심으로 재정의한다. 이 경우에도 1차 가설의 형식은 유지된다.

B3 가 19/22 로 살아남았으므로 **B 재정의는 하지 않는다.** 세 축 구성을 유지하고 B2 만 보조 관찰로 내린다.

- 원인: TDC 가 벤치마크 구축 시 탈염을 일괄 적용했다. 실측 최대치가 `bbb_martins` 5.17%(105개),
  `cyp2d6_veith` 4.18% 이고 여러 물성은 0% 다. 계획서 5.3절이 짝이온 임의 부착을 금지하므로 만들어낼 수도 없다.
- **게이트 임계값 10% 는 사전 지정값이므로 낮추지 않는다.**
- 계획서 5.8절의 **통합 B 는 세 축(B1 호변 · B1 양성자 · B3 입체)의 백분위 최댓값**으로 산출한다.
- B2 는 염 등록 분자가 존재하는 물성(`bbb_martins` 105개 등)에서 기술적 관찰로만 보고한다.

#### B3 표본부족 3종의 정체 — 데이터셋 단위 정보 결손

| 물성 | 입체 표기 보유 | 표기 없는 분자 중 미지정 입체중심 보유 (표본 300) |
|---|---|---|
| `dili` | 0.00% | **56.3%** |
| `ld50_zhu` | 0.00% | 26.7% |
| `solubility_aqsoldb` | 9.81% | 21.3% |

`dili` 는 승인 약물 475개인데 표기가 **하나도** 없고, 그런데도 절반 이상이 입체중심을 가진다.
화학적 성질이 아니라 **큐레이션 과정에서 입체 정보가 통째로 제거된 것**이다.
계획서 4.1절의 "B3 는 실험 데이터의 정보 결손에서 비롯된다"가 분자 단위가 아니라
데이터셋 단위로 일어난 사례이며, RQ1 특성화 결과로 보고할 가치가 있다.

**측정 불가는 원리적이다.** B3 는 "표기 유지 vs 제거"인데 제거할 표기가 없다.
반대 방향(표기 추가)은 계획서 3.2절이 금지한다 — 미표기 분자에 입체를 붙이는 것은
이성질체를 고르는 행위이고, 그건 다른 화합물을 만드는 것이기 때문이다.

`solubility_aqsoldb` 는 9.81% 로 **0.19%p 차이로 탈락**했다. 팀원 전수 계산이라 표본오차가 아니며
사전 지정 임계값에 따라 탈락이 맞지만, 보고 시 경계 사례임을 각주로 밝힌다.

## 5. 태스크 8 — 4중 분할

> ### 🚨 순서를 틀리면 전체 결과가 무효가 된다
> **(1) 부모 분자 확정 → (2) 골격 기준으로 분할 → (3) 그 다음에 변형 생성**
>
> (1)은 팀원이, (2)는 지금 여기서. 변형을 분할보다 먼저 만들면
> 같은 화합물의 다른 형태가 학습 집합과 시험 집합에 나뉘어 들어가 **정보가 새어나간다**.
> 이 노트북은 (2)까지만 하고 **변형은 만들지 않는다**.

| 이름 | 역할 |
|---|---|
| `train` | 모델 학습 |
| `calib` | 컨포멀 보정 — 예측 구간 폭 결정 |
| `meta`  | 메타 보정 — 신호 결합 규칙·임계값 결정 |
| `test`  | 최종 시험 — 골격 기반, **단 한 번만** 사용 |

### 분할 모드 — 기본값은 `full_rescaffold`

| 모드 | 내용 |
|---|---|
| **`full_rescaffold`** (기본) | 중복 제거 후 전체를 우리 골격 기준으로 4중 분할 |
| `tdc_test_preserved` | TDC 공식 test 를 그대로 최종 시험 집합으로 두고 train_val 만 셋으로 쪼갠다 |

처음에는 공개 벤치마크와 비교 가능한 `tdc_test_preserved` 를 기본값으로 두었으나,
실측 결과 **TDC 공식 분할이 우리 골격 정의 기준으로는 분리되어 있지 않았다.**

| 물성 | 가로지르는 골격 그룹 | 버려질 train_val 행 |
|---|---|---|
| `solubility_aqsoldb` | 13 | 2,089 (**26.2%**) |
| `bbb_martins` | 2 | 148 (9.1%) |
| `cyp2c9_veith` | 14 | 610 (6.3%) |
| `herg` | 1 | 32 (6.1%) |
| `cyp3a4_veith` | 10 | 594 (6.0%) |

그룹 수는 적은데 행 수가 많다 = 벤젠·피리딘 같은 **범용 골격**이 양쪽에 걸쳐 있다는 뜻이다.
누출을 막으려면 train_val 쪽을 버려야 하는데, 그러면 학습 데이터의 1/4까지 사라진다.

애초에 계획서 5.7절은 **4중 분할**을 요구하고 TDC는 2분할만 제공하므로 공식 프로토콜은 이미 벗어나 있고,
6장의 평가 지표(AURC·오차 탐지)도 TDC 리더보드 점수와 비교하지 않는다.
따라서 비교 가능성을 지키려고 데이터를 버릴 이유가 없다.
**우리가 직접 쪼개면 골격 분리가 구성상 보장되어 버려지는 행이 0이 된다.**
22종에 동일한 프로토콜이 적용된다는 이점도 있다.

`tdc_test_preserved` 로 되돌리려면 아래 셀의 `SPLIT_MODE` 만 바꾸면 된다
(그 경우 `OVERLAP_POLICY` 가 적용되어 위 표만큼 행이 버려진다).

**고리 없는 분자**는 골격이 빈 값이다. 전부 한 그룹으로 묶으면 거대 덩어리가 생겨 분할이 망가지므로
기본값은 분자당 독립 그룹(`ACYCLIC_AS_SINGLETON = True`).

### 📌 사전 확정 사항 — 고리 유무로 층을 나눠 배정한다 (`STRATIFY_ACYCLIC = True`, 2026-08-17 확정)

골격 그룹을 **큰 것부터** 배정하면 train 이 큰 고리 골격으로 먼저 채워지고,
고리 없는 분자(전부 단일 그룹)가 작은 세 집합으로 밀려난다. 실측 결과는 이랬다.

| 물성 | train | calib | meta | test |
|---|---|---|---|---|
| `solubility_aqsoldb` | 10.7% | 66.9% | 68.5% | 64.7% |
| `ld50_zhu` | 11.4% | 62.8% | 61.4% | 58.6% |
| `ames` | 6.3% | 43.4% | 44.1% | 42.3% |

**왜 문제인가.** 고리 없는 작은 분자는 호변이성질체·양성자화 상태가 적어 **B 가 작게** 나오는 동시에,
학습에서 못 본 유형이라 **오차는 크게** 난다. "B 작음 + 오차 큼" 분자가 시험 집합의 절반을 차지하면
1차 가설(B 가 클수록 오차가 크다)과 **반대 방향의 교란**이 생긴다.
가설이 기각돼도 B 때문인지 조성 인공물 때문인지 구분할 수 없다.

**해결.** 고리없음 / 고리있음 두 층에서 각각 70/10/10/10 을 맞춘다.
고리 없는 분자는 어차피 전부 서로 다른 단일 그룹이므로 **골격 분리는 그대로 보장**된다.
바뀌는 것은 분자 유형 조성뿐이며, 의도한 분포 이동(신규 골격)은 손대지 않는다.

| 지표 | `True` (채택) | `False` |
|---|---|---|
| train vs test 고리없음 최대 차이 | **1.1%p** | 54.0%p |
| calib vs test 최대 차이 | **1.9%p** | 10.4%p |
| 10%p 이상 벌어지는 물성 | 없음 | train 4종 / calib 1종 |

계층화 후 각 집합의 고리없음 비율은 모집단 비율과 일치한다
(`solubility_aqsoldb` 27.5%, `ld50_zhu` 26.3%). 시험 골격의 신규성은 여전히 100% 다.
`calib` ≈ `test` 가 더 잘 맞으므로 컨포멀 교환 가능성(계획서 3.3절)도 함께 개선됐다.

**남는 단서**: `ld50_zhu`(26.3%) 와 `solubility_aqsoldb`(27.5%) 는 실제로 고리 없는 분자가 많다.
이제 네 집합에 균등히 퍼져 있지만, 그 두 물성에서는 "신규 골격 = 어렵다"는 전제가 약하므로
RQ1 해석에 각주를 단다.

In [ ]:
# ============================================================================
# [셀 10] 골격 그룹 키 만들기
# ============================================================================
ACYCLIC_AS_SINGLETON = True

for name, d in tables.items():
    scaf = d.scaffold.fillna("")                       # 고리 없는 분자 = 빈 문자열
    d["is_acyclic"] = scaf == ""
    if ACYCLIC_AS_SINGLETON:
        d["scaffold_group"] = np.where(d.is_acyclic, "__ACYCLIC__" + d.row_uid, scaf)
    else:
        d["scaffold_group"] = np.where(d.is_acyclic, "__ACYCLIC__", scaf)
    tables[name] = d

rows = []
for name, d in tables.items():
    k = d[d.is_kept]
    g = k.groupby("scaffold_group").size()
    rows.append({"dataset": name, "n_kept": len(k), "n_groups": int(g.size),
                 "최대그룹_pct": round(100 * g.max() / max(len(k), 1), 2),
                 "단일분자그룹_pct": round(100 * (g == 1).sum() / max(g.size, 1), 2),
                 "고리없음_pct": round(100 * k.is_acyclic.mean(), 2)})
scaffold_report = pd.DataFrame(rows).sort_values("최대그룹_pct", ascending=False)
display(scaffold_report)
risky = scaffold_report[scaffold_report.최대그룹_pct > 10]
if len(risky):
    print("[주의] 단일 골격이 10% 를 넘는 물성 — 분할 비율이 목표에서 벗어날 수 있다:")
    print(risky.to_string(index=False))

,dataset,n_kept,n_groups,최대그룹_pct,단일분자그룹_pct,고리없음_pct
16,ld50_zhu,7342,3606,20.95,87.35,26.29
20,solubility_aqsoldb,9576,4524,18.26,86.89,27.46
0,ames,7255,2836,15.64,77.47,17.37
12,dili,475,344,11.37,87.21,7.37
15,hia_hou,578,403,11.07,85.11,3.29
13,half_life_obach,665,458,8.72,82.31,2.26
8,cyp2d6_substrate_carbonmangels,663,470,8.14,83.83,3.17
6,cyp2c9_substrate_carbonmangels,666,471,8.11,83.65,3.15
10,cyp3a4_substrate_carbonmangels,667,473,7.95,83.30,3.15
2,bioavailability_ma,640,456,7.66,83.99,2.66


[주의] 단일 골격이 10% 를 넘는 물성 — 분할 비율이 목표에서 벗어날 수 있다:
           dataset  n_kept  n_groups  최대그룹_pct  단일분자그룹_pct  고리없음_pct
          ld50_zhu    7342      3606     20.95       87.35     26.29
solubility_aqsoldb    9576      4524     18.26       86.89     27.46
              ames    7255      2836     15.64       77.47     17.37
              dili     475       344     11.37       87.21      7.37
           hia_hou     578       403     11.07       85.11      3.29


In [ ]:
# ============================================================================
# [셀 11] 골격 그룹 분할 함수
# ============================================================================
import hashlib
from collections import defaultdict

def greedy_scaffold_split(group_ids, fractions, seed=SEED):
    """골격 그룹 단위로 행을 나눈다.

    group_ids : 행마다의 골격 그룹 키 (리스트/Series)
    fractions : {"train":0.7, ...} 합이 1
    return    : 행마다의 집합 이름 리스트
    """
    groups = defaultdict(list)
    for i, g in enumerate(group_ids):
        groups[g].append(i)

    n_total = len(group_ids)
    names = list(fractions)
    target = {k: fractions[k] * n_total for k in names}
    count = {k: 0 for k in names}

    def _tiebreak(g):
        # 같은 크기 그룹의 순서를 시드에 따라 결정론적으로 섞는다
        return hashlib.md5(f"{seed}:{g}".encode()).hexdigest()

    # 큰 그룹부터 배정해야 비율이 잘 맞는다
    order = sorted(groups.items(), key=lambda kv: (-len(kv[1]), _tiebreak(kv[0])))

    assign = [None] * n_total
    for g, idxs in order:
        # '아직 목표에 가장 많이 못 미친' 집합에 통째로 넣는다
        k = max(names, key=lambda k: (target[k] - count[k], -names.index(k)))
        for i in idxs:
            assign[i] = k
        count[k] += len(idxs)
    return assign


# ---- 함수 동작 확인 1: 골격이 잘게 흩어진 정상적인 경우 -> 비율이 목표대로 나온다 ----
FRACTIONS_DEMO = {"train": .7, "calib": .1, "meta": .1, "test": .1}
ok_groups = [f"g{i // 2}" for i in range(2000)]      # 골격 1000종, 각 2분자
ok = pd.Series(greedy_scaffold_split(ok_groups, FRACTIONS_DEMO))
print("[정상 사례] 실제 비율:", (ok.value_counts(normalize=True).round(3)).to_dict())

# ---- 함수 동작 확인 2: 거대 골격이 하나 있는 경우 -> 비율이 목표에서 벗어난다 ----
skew_groups = ["A"] * 50 + ["B"] * 30 + ["C"] * 10 + [f"S{i}" for i in range(10)]
skew = pd.Series(greedy_scaffold_split(skew_groups, FRACTIONS_DEMO))
print("[쏠린 사례] 실제 비율:", (skew.value_counts(normalize=True).round(3)).to_dict())
print("  -> 그룹을 쪼갤 수 없으므로 거대 골격이 있으면 비율이 어긋난다.")
print("     (노트북 01의 04_scaffold_report.csv 에서 largest_group_pct 로 미리 확인한 그 문제)")

print("\n그룹이 두 집합에 걸치는가:",
      any(len(set(np.array(skew)[np.array(skew_groups) == g])) > 1 for g in set(skew_groups)))

[정상 사례] 실제 비율: {'train': 0.7, 'test': 0.1, 'calib': 0.1, 'meta': 0.1}
[쏠린 사례] 실제 비율: {'train': 0.8, 'calib': 0.1, 'test': 0.05, 'meta': 0.05}
  -> 그룹을 쪼갤 수 없으므로 거대 골격이 있으면 비율이 어긋난다.
     (노트북 01의 04_scaffold_report.csv 에서 largest_group_pct 로 미리 확인한 그 문제)

그룹이 두 집합에 걸치는가: False


In [ ]:
# ============================================================================
# [셀 12] 22종 4중 분할 실행
# ============================================================================
from sklearn.model_selection import GroupKFold

SPLIT_MODE     = "full_rescaffold"       # 또는 "tdc_test_preserved" (위 마크다운 참조)
OVERLAP_POLICY = "drop_from_trainval"    # tdc_test_preserved 모드에서만 쓰임
FRACTIONS      = {"train": 0.70, "calib": 0.10, "meta": 0.10, "test": 0.10}
N_CV_FOLDS     = 5
MIN_META_ROWS  = 200

# 고리없음/고리있음을 따로 배정해 집합 간 조성을 맞출지
#   False 로 두면: 큰 골격 그룹이 train 을 먼저 채우고 고리 없는 분자(전부 단일 그룹)가
#                  작은 세 집합으로 밀려난다. 실측 예: solubility_aqsoldb 의 고리없음 비율이
#                  train 10.7% vs test 64.7% 로 6배 벌어졌다.
#   True  로 두면: 두 층에서 각각 70/10/10/10 을 맞춘다. 골격 분리는 그대로 유지되고
#                  (고리 없는 분자는 어차피 전부 서로 다른 단일 그룹),
#                  의도한 분포 이동(신규 골격)만 남고 분자 유형 차이는 사라진다.
STRATIFY_ACYCLIC = True


def assign_splits(sub, fractions):
    """골격 그룹 단위 배정. STRATIFY_ACYCLIC 이면 고리 유무로 층을 나눠 각각 배정한다."""
    out = pd.Series(index=sub.index, dtype=object)
    if STRATIFY_ACYCLIC:
        strata = [sub.index[~sub.is_acyclic], sub.index[sub.is_acyclic]]
    else:
        strata = [sub.index]
    for idx in strata:
        if len(idx) == 0:
            continue
        out.loc[idx] = greedy_scaffold_split(
            sub.loc[idx, "scaffold_group"].tolist(), fractions, seed=SEED)
    return out

splits, split_rows = {}, []

for name in tqdm(DATASETS, desc="4중 분할"):
    d = tables[name][tables[name].is_kept].reset_index(drop=True)
    n_dropped_overlap = 0

    if SPLIT_MODE == "tdc_test_preserved":
        is_test = d.tdc_split == "test"
        overlap = set(d.loc[is_test, "scaffold_group"]) & set(d.loc[~is_test, "scaffold_group"])
        if overlap and OVERLAP_POLICY == "drop_from_trainval":
            bad = (~is_test) & d.scaffold_group.isin(overlap)
            n_dropped_overlap = int(bad.sum())
            d = d[~bad].reset_index(drop=True)
            is_test = d.tdc_split == "test"

        d["split"] = None
        d.loc[is_test, "split"] = "test"
        tv = d.index[~is_test]
        sub = {k: FRACTIONS[k] for k in ("train", "calib", "meta")}
        s = sum(sub.values())
        sub = {k: v / s for k, v in sub.items()}          # train_val 안에서 재정규화
        d.loc[tv, "split"] = assign_splits(d.loc[tv], sub)
    else:
        d["split"] = assign_splits(d, FRACTIONS)

    # 표본이 작은 물성용: test 를 제외한 영역에 골격 그룹 교차적합 폴드를 미리 부여
    d["cv_fold"] = -1
    nontest = d.index[d.split != "test"]
    if len(nontest) and d.loc[nontest, "scaffold_group"].nunique() >= N_CV_FOLDS:
        gkf = GroupKFold(n_splits=N_CV_FOLDS)
        sub_df = d.loc[nontest]
        for f, (_, va) in enumerate(gkf.split(sub_df, groups=sub_df.scaffold_group)):
            d.loc[sub_df.index[va], "cv_fold"] = f

    splits[name] = d

    y = pd.to_numeric(d.Y_final, errors="coerce")
    rec = {"dataset": name, "task_type": TASK_TYPE_REF[name],
           "n_total": len(d), "n_dropped_scaffold_overlap": n_dropped_overlap}
    for k in ["train", "calib", "meta", "test"]:
        msk = d.split == k
        rec[f"n_{k}"] = int(msk.sum())
        rec[f"pct_{k}"] = round(100 * msk.mean(), 1)
        rec[f"y_{k}"] = round(float(y[msk].mean()), 3) if msk.sum() else np.nan
        # 고리 없는 분자 비율: 이 값이 높으면 "신규 골격"이라는 말의 의미가 약해진다.
        #   (에탄올 vs 프로판올처럼 골격이 없어 자동으로 신규 골격으로 잡히는 분자들)
        rec[f"acyclic_{k}"] = round(100 * d.loc[msk, "is_acyclic"].mean(), 1) if msk.sum() else np.nan
    rec["use_cv_for_meta"] = rec["n_meta"] < MIN_META_ROWS
    split_rows.append(rec)

split_summary = pd.DataFrame(split_rows)
display(split_summary[["dataset", "task_type", "n_total", "n_train", "n_calib", "n_meta", "n_test",
                       "pct_train", "pct_calib", "pct_meta", "pct_test",
                       "acyclic_train", "acyclic_test",
                       "n_dropped_scaffold_overlap", "use_cv_for_meta"]])

# 고리 없는 분자가 많은 물성은 골격 분할이 만드는 분포 이동이 약하다 -> 결과 해석에 단서 필요
hi = split_summary[split_summary.acyclic_test > 20]
if len(hi):
    print("[주의] 시험 집합의 20% 이상이 '고리 없는 분자'인 물성 — 신규 골격의 난이도가 과대평가된다:")
    print(hi[["dataset", "n_test", "acyclic_test"]].to_string(index=False))

# 조성 점검 두 가지
#   (1) train vs test : 벌어지면 '분자 유형'이라는 의도치 않은 교란이 생긴다
#   (2) calib vs test : 벌어지면 컨포멀 교환 가능성(계획서 3.3절)이 흔들린다
gap_tr = (split_summary.acyclic_train - split_summary.acyclic_test).abs()
gap_cal = (split_summary.acyclic_calib - split_summary.acyclic_test).abs()
print()
print(f"고리없음 비율 차이 — train vs test 최대 {gap_tr.max():.1f}%p / calib vs test 최대 {gap_cal.max():.1f}%p")
for label, gap in [("train vs test", gap_tr), ("calib vs test", gap_cal)]:
    bad = split_summary.loc[gap >= 10, "dataset"].tolist()
    print(f"  {label:14s} 10%p 이상 벌어지는 물성: {bad or '없음'}")

4중 분할:   0%|          | 0/22 [00:00<?, ?it/s]

,dataset,task_type,n_total,n_train,n_calib,n_meta,n_test,pct_train,pct_calib,pct_meta,pct_test,acyclic_train,acyclic_test,n_dropped_scaffold_overlap,use_cv_for_meta
0,ames,classification,7255,5079,726,725,725,70.0,10.0,10.0,10.0,17.4,17.4,0,False
1,bbb_martins,classification,1955,1369,196,195,195,70.0,10.0,10.0,10.0,4.9,4.6,0,True
2,bioavailability_ma,classification,640,448,65,64,63,70.0,10.2,10.0,9.8,2.7,1.6,0,True
3,caco2_wang,regression,905,634,91,90,90,70.1,10.1,9.9,9.9,2.8,2.2,0,True
4,clearance_hepatocyte_az,regression,1020,714,103,102,101,70.0,10.1,10.0,9.9,0.4,0.0,0,True
5,clearance_microsome_az,regression,1102,772,110,110,110,70.1,10.0,10.0,10.0,0.1,0.0,0,True
6,cyp2c9_substrate_carbonmangels,classification,666,466,67,67,66,70.0,10.1,10.1,9.9,3.2,3.0,0,True
7,cyp2c9_veith,classification,12047,8433,1205,1205,1204,70.0,10.0,10.0,10.0,2.2,2.2,0,False
8,cyp2d6_substrate_carbonmangels,classification,663,465,66,66,66,70.1,10.0,10.0,10.0,3.2,3.0,0,True
9,cyp2d6_veith,classification,13089,9163,1310,1308,1308,70.0,10.0,10.0,10.0,2.1,2.1,0,False


[주의] 시험 집합의 20% 이상이 '고리 없는 분자'인 물성 — 신규 골격의 난이도가 과대평가된다:
           dataset  n_test  acyclic_test
          ld50_zhu     734          26.3
solubility_aqsoldb     957          27.5

고리없음 비율 차이 — train vs test 최대 1.1%p / calib vs test 최대 1.9%p
  train vs test  10%p 이상 벌어지는 물성: 없음
  calib vs test  10%p 이상 벌어지는 물성: 없음


In [ ]:
# ============================================================================
# [셀 13] 🚨 누출 검사 — 여기서 하나라도 걸리면 이후 결과는 전부 무효다
# ============================================================================
problems = []

for name, d in splits.items():
    # (1) 같은 골격 그룹이 두 집합에 걸치면 안 된다
    per_group = d.groupby("scaffold_group")["split"].nunique()
    if (per_group > 1).any():
        problems.append(f"[{name}] 골격 그룹이 여러 집합에 걸침: {int((per_group > 1).sum())}개")

    # (2) 같은 부모 분자가 두 집합에 있으면 안 된다 (중복 제거가 제대로 됐는지 재확인)
    per_parent = d.groupby("parent_smiles")["split"].nunique()
    if (per_parent > 1).any():
        problems.append(f"[{name}] 동일 부모 분자가 여러 집합에 존재: {int((per_parent > 1).sum())}개")

    # (3) 빈 집합이 없어야 한다
    for k in ["train", "calib", "meta", "test"]:
        if (d.split == k).sum() == 0:
            problems.append(f"[{name}] '{k}' 집합이 비어 있음")

    # (4) 분류 물성은 각 집합에 양성/음성이 모두 있어야 평가가 가능하다
    if TASK_TYPE_REF[name] == "classification":
        y = pd.to_numeric(d.Y_final, errors="coerce")
        for k in ["train", "calib", "meta", "test"]:
            m = d.split == k
            if m.sum() and (y[m].nunique() < 2 or min((y[m] == 1).sum(), (y[m] == 0).sum()) < 5):
                problems.append(f"[{name}] '{k}' 집합의 클래스가 한쪽으로 쏠림 "
                                f"(양성 {int((y[m] == 1).sum())} / 음성 {int((y[m] == 0).sum())})")

print("=" * 78)
if problems:
    print(f"문제 {len(problems)}건 발견:")
    for p in problems:
        print("  -", p)
    print("\n대응: 시드(SEED)를 바꾸거나, 작은 물성은 meta 를 cv_fold 로 대체하세요.")
else:
    print("✅ 누출 검사 통과 — 골격/부모분자가 집합을 가로질러 존재하지 않습니다.")
print("=" * 78)

# 시험 집합의 골격이 학습 집합에 하나도 없다는 것을 수치로도 확인
print("\n학습 골격 대비 시험 골격의 신규성 (설계상 100% 여야 정상)")
for name, d in list(splits.items())[:5]:
    tr = set(d.loc[d.split == "train", "scaffold_group"])
    te = set(d.loc[d.split == "test", "scaffold_group"])
    print(f"  {name:32s} 시험 골격 {len(te):5d}개 중 학습에 없던 것 "
          f"{100 * len(te - tr) / max(len(te), 1):5.1f}%")

✅ 누출 검사 통과 — 골격/부모분자가 집합을 가로질러 존재하지 않습니다.

학습 골격 대비 시험 골격의 신규성 (설계상 100% 여야 정상)
  ames                             시험 골격   488개 중 학습에 없던 것 100.0%
  bbb_martins                      시험 골격   195개 중 학습에 없던 것 100.0%
  bioavailability_ma               시험 골격    63개 중 학습에 없던 것 100.0%
  caco2_wang                       시험 골격    84개 중 학습에 없던 것 100.0%
  clearance_hepatocyte_az          시험 골격   101개 중 학습에 없던 것 100.0%


In [ ]:
# ============================================================================
# [셀 14] 저장 — 팀원 파일은 건드리지 않고 새 이름으로만 쓴다
# ============================================================================
MOL_COLS = ["row_uid", "dataset", "task_type", "Drug_ID", "smiles_original", "parent_smiles",
            "Y", "Y_final", "tdc_split", "valid", "has_salt", "has_stereo",
            "n_tautomers", "n_protomers", "scaffold", "is_acyclic", "scaffold_group",
            "dup_group_size", "dup_role", "is_kept"]
SPLIT_COLS = ["row_uid", "dataset", "task_type", "Drug_ID", "smiles_original", "parent_smiles",
              "Y_final", "scaffold", "scaffold_group", "tdc_split", "split", "cv_fold"]

for name in DATASETS:
    (PROC_DIR / name).mkdir(parents=True, exist_ok=True)   # 출력 폴더는 우리가 만든다
    tables[name][MOL_COLS].to_csv(PROC_DIR / name / "molecules_labeled.csv", index=False)
    splits[name][SPLIT_COLS].to_csv(PROC_DIR / name / "splits.csv", index=False)

join_report.to_csv(REPORT_DIR / "02_join_report.csv", index=False)
dedup_report.to_csv(REPORT_DIR / "03_dedup_report.csv", index=False)
scaffold_report.to_csv(REPORT_DIR / "04_scaffold_report.csv", index=False)
split_summary.assign(split_mode=SPLIT_MODE, seed=SEED).to_csv(
    REPORT_DIR / "08_split_summary.csv", index=False)

print("저장 완료")
print(f"  <물성명>/molecules_labeled.csv , <물성명>/splits.csv")
print(f"  {REPORT_DIR} 아래 02·03·04·05·06·07·08 보고표")
print(f"\n총 {split_summary.n_total.sum():,}분자")
print(split_summary[["n_train", "n_calib", "n_meta", "n_test"]].sum().to_string())
small = split_summary[split_summary.use_cv_for_meta]
if len(small):
    print(f"\n[알림] meta 가 {MIN_META_ROWS}건 미만인 물성 {len(small)}종 — cv_fold 기반 폴드 외 예측을 쓰세요:")
    print("  " + ", ".join(small.dataset))

저장 완료
  <물성명>/molecules_labeled.csv , <물성명>/splits.csv
  /content/drive/MyDrive/MIST/outputs/processed/reports 아래 02·03·04·05·06·07·08 보고표

총 79,903분자
n_train    55937
n_calib     8001
n_meta      7986
n_test      7979

[알림] meta 가 200건 미만인 물성 15종 — cv_fold 기반 폴드 외 예측을 쓰세요:
  bbb_martins, bioavailability_ma, caco2_wang, clearance_hepatocyte_az, clearance_microsome_az, cyp2c9_substrate_carbonmangels, cyp2d6_substrate_carbonmangels, cyp3a4_substrate_carbonmangels, dili, half_life_obach, herg, hia_hou, pgp_broccatelli, ppbr_az, vdss_lombardo


## 6. 다음 단계 — 변형 생성 순서 가드

```text
[완료] 1. 부모 분자 확정   (팀원 molecule_profile.csv)
[완료] 2. 골격 기준 분할   (지금 여기)
[다음] 3. 변형 생성        <- 반드시 각 집합 안에서 따로
```

- 변형본은 원본 행의 `split` 값을 **그대로 물려받는다**. 변형 후 다시 분할하면 안 된다.
- 어떤 축을 만들지는 `reports/07_axis_decision.csv` 의 `use_*` 가 `사용` 인 축만.

In [ ]:
# ============================================================================
# [셀 15] 변형 생성 단계에서 쓸 안전장치 + 다음 단계 뼈대
# ============================================================================
def assert_ready_for_variants(df, dataset_name):
    """변형 생성 직전에 호출하는 검사기. 분할이 끝난 표가 아니면 예외를 던진다."""
    if "split" not in df.columns:
        raise RuntimeError(
            f"[{dataset_name}] 'split' 열이 없습니다. 변형 생성은 골격 분할 이후에만 허용됩니다. "
            "(계획서 5.7절 — 순서를 어기면 정보 누출)")
    if df.groupby("scaffold_group")["split"].nunique().gt(1).any():
        raise RuntimeError(f"[{dataset_name}] 골격 그룹이 여러 집합에 걸쳐 있습니다. 분할을 다시 하세요.")
    if df.groupby("parent_smiles")["split"].nunique().gt(1).any():
        raise RuntimeError(f"[{dataset_name}] 동일 부모 분자가 여러 집합에 있습니다. 중복 제거를 다시 하세요.")
    return True


# 예시: 모든 물성에서 가드가 통과하는지 확인
for name, d in splits.items():
    assert_ready_for_variants(d, name)
print("✅ 22종 모두 변형 생성 준비 완료 (분할이 선행되었음을 확인)")

# ---------------------------------------------------------------------------
# 다음 단계(이 노트북 범위 밖)의 뼈대 — 실행하지 않고 형태만 남겨둔다
# ---------------------------------------------------------------------------
PSEUDOCODE = """
for 물성 in 22종:
    df = read splits.csv
    assert_ready_for_variants(df, 물성)              # <- 순서 보증
    axes = 07_axis_decision.csv 에서 use_* == "사용" 인 축
    for row in df.itertuples():                      # 행 단위로
        variants = []
        if "B1_tautomer"    in axes: variants += 호변이성질체_열거(row.parent_smiles)
        if "B1_protonation" in axes: variants += 양성자화_열거(row.parent_smiles, 물성별_pH)
        if "B2_salt"        in axes: variants += [row.smiles_original, row.parent_smiles]
        if "B3_stereo"      in axes: variants += [row.parent_smiles, 입체제거(row.parent_smiles)]
        for v in variants:
            for k in range(N_RANDOM_SMILES):         # A(표현 불안정성)용 등가 SMILES
                yield dict(row_uid=row.row_uid, split=row.split,   # <- split 을 그대로 물려받는다
                           axis=..., state_id=..., smiles=무작위화(v))
"""
print(PSEUDOCODE)

✅ 22종 모두 변형 생성 준비 완료 (분할이 선행되었음을 확인)

for 물성 in 22종:
    df = read splits.csv
    assert_ready_for_variants(df, 물성)              # <- 순서 보증
    axes = 07_axis_decision.csv 에서 use_* == "사용" 인 축
    for row in df.itertuples():                      # 행 단위로
        variants = []
        if "B1_tautomer"    in axes: variants += 호변이성질체_열거(row.parent_smiles)
        if "B1_protonation" in axes: variants += 양성자화_열거(row.parent_smiles, 물성별_pH)
        if "B2_salt"        in axes: variants += [row.smiles_original, row.parent_smiles]
        if "B3_stereo"      in axes: variants += [row.parent_smiles, 입체제거(row.parent_smiles)]
        for v in variants:
            for k in range(N_RANDOM_SMILES):         # A(표현 불안정성)용 등가 SMILES
                yield dict(row_uid=row.row_uid, split=row.split,   # <- split 을 그대로 물려받는다
                           axis=..., state_id=..., smiles=무작위화(v))



---

## ✅ 태스크 3·7·8 완료

### 우리가 만든 파일 (팀원 파일은 그대로 둠)
| 경로 | 내용 |
|---|---|
| `<물성명>/molecules_labeled.csv` | 팀 프로필 + `Y`/`tdc_split` + 중복 판정 |
| `<물성명>/splits.csv` | **4중 분할** (`split`, `cv_fold`) |
| `reports/02_join_report.csv` | 조인 검증 |
| `reports/03_dedup_report.csv` | 중복·라벨충돌 제거 내역 |
| `reports/04_scaffold_report.csv` | 골격 통계 |
| `reports/05_variation_gate.csv` | 게이트 (팀 수치 재사용, B1 두 축 분리) |
| `reports/06_transformation_allowance.csv` | **허용성 표 — 수작업 검토 대상** |
| `reports/07_axis_decision.csv` | 게이트 × 허용성 = 최종 사용 축 |
| `reports/08_split_summary.csv` | 분할 요약 |

### 팀원 파일 (읽기만 함)
`<물성명>/molecule_profile.csv`, `prescreen_summary.{csv,json}`, `gate_decision_final.csv`, `a_axis_*.json`

### 남은 수작업
`reports/06_transformation_allowance.csv` 의 `reviewed` 를 22종 모두 `True` 로.
계획서 5.2절의 "결과 관찰 이전 고정"은 그때 충족된다.